# Walkthrough — Clean Pipeline 56-32 (Rafif): Phase A → B → C + Verdict

**100% Self-Contained Notebook for Google Colab, Kaggle, and Local Execution.**
All pipeline code (`protocol.json`, `prepare_windows.py`, `split_scale.py`, `cluster_evaluate.py`, `check_contract.py`) is embedded directly in this notebook so it can execute end-to-end anywhere without cloning external repositories.

Locked protocol: `clean-pipeline-56-32/protocol.json` (tumbling 60/60, 13 channels, chrono 70/30 + 1h embargo, train-only scaling, collapse/NO-WINNER guards).

| Phase | Goal | Cell outputs | Paper relevance |
|---|---|---|---|
| Contract | Prove the machinery on synthetic data (no dataset) | PASS line | Methods credibility |
| A. Windows | 60-s tumbling, gap-aware, coverage-gated windows | manifest + 38.177 valid windows | Bab II temporal design |
| B. Split+Scale | Chrono split, embargo, train-only scalers, train-only EDA | manifests + scaler joblibs + EDA figs | Bab II validation design |
| C. Evaluate | 126-run grid (3 scalers × 2 algos × 7 k × 3 seeds) | verdict_ledger + summary + figs | Bab III robustness |
| Recap | Verdict analysis + MISSING-check + export | verdict table | Appendix |

> **Setup:** raw CSV resolved explicitly below (repo `dataset/`, Kaggle Input, or kagglehub). Phase C takes belasan menit CPU (126 fits incl. GMM) — Contract + A + B run in minutes.


#### Cell: Environment & dependencies (reproducibility record)
**What:** installs `kagglehub` if missing, prepares working directory structure, and records runtime versions.
**Why:** a run without recorded versions is not reproducible; this cell is the record.


In [ ]:
import os, sys
from pathlib import Path

# In local repo, switch from notebooks/ to repo root if needed
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'clean-pipeline-56-32').exists():
    os.chdir(Path.cwd().parent)

# Ensure pipeline destination directory structure exists
Path('clean-pipeline-56-32/scripts').mkdir(parents=True, exist_ok=True)

try:
    import kagglehub
except ImportError:
    print('installing kagglehub ...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub[pandas-datasets]'])

import sklearn, pandas, numpy, joblib
print('python:', sys.version.split()[0])
print('sklearn:', sklearn.__version__, '| pandas:', pandas.__version__, '| numpy:', numpy.__version__, '| joblib:', joblib.__version__)
print('working dir:', Path.cwd())


## Embedded Clean Pipeline Machinery
The following cells write the locked protocol specification and all phase modules directly to `clean-pipeline-56-32/`.
This makes the notebook completely self-contained in Colab and Kaggle sessions.


### Specification: `clean-pipeline-56-32/protocol.json`
Defines the locked 56-32 protocol: 13 channels, tumbling 60/60, SPP row drop, sentinel masking, 70/30 chrono split, 1h embargo, and NO-WINNER rules.


In [ ]:
%%writefile clean-pipeline-56-32/protocol.json
{
  "protocol_version": "1.0",
  "well": "56-32",
  "phase_a": {
    "input_frequency_hz": 1,
    "read_mode": "pandas_csv_chunked",
    "chunksize": 50000,
    "numeric_dtype": "float64",
    "timestamp_format": "%Y/%m/%d %H:%M:%S",
    "drop_rule": "SPP <= -900",
    "sentinel": -999.25,
    "sentinel_action": "mask_sensor_values_to_NaN_after_SPP_row_drop",
    "segment_reset": "timestamp delta after row-drop is not exactly +1 second",
    "window_seconds": 60,
    "stride_seconds": 60,
    "overlap_percent": 0,
    "window_anchor": "first retained row of each segment",
    "tail_policy": "discard_partial_segment_tail",
    "channels": 13,
    "coverage_minimum_per_channel": 54,
    "features_per_channel": ["mean", "population_std_ddof_0", "endpoint_delta_last_finite_minus_first_finite"],
    "features_total": 39
  },
  "phase_b": {
    "split": "chronological",
    "train_fraction": 0.7,
    "test_fraction": 0.3,
    "embargo_seconds_minimum": 3600,
    "scaler_fit_scope": "train_only",
    "candidate_scalers": ["StandardScaler", "RobustScaler", "MinMaxScaler"]
  },
  "phase_c": {
    "models": ["KMeans", "GaussianMixture"],
    "k_values": [2, 3, 4, 5, 6, 7, 8],
    "fit_scope": "train_only",
    "evaluate_separately": ["train", "test"],
    "metrics": ["silhouette", "davies_bouldin", "occupancy", "collapse"],
    "unstable_or_collapsed_outcome": "NO-WINNER",
    "cluster_label_format": "C0..Ck",
    "physical_regime_claim": "requires_driller_log_or_independent_domain_validation"
  }
}


### Phase A Module: `clean-pipeline-56-32/scripts/prepare_windows.py`
Implements SPP row dropping, sensor sentinel masking, proxy derivation (`Hydraulic_P`, `Mech_P`), 1-second gap reset, coverage gating (≥54/60), and 39 feature computation.


In [ ]:
%%writefile clean-pipeline-56-32/scripts/prepare_windows.py
#!/usr/bin/env python3
"""Build clean 60-second tumbling windows from the Well 56-32 raw CSV."""
from __future__ import annotations

import argparse
import hashlib
import json
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

CHUNKSIZE = 50_000
WINDOW_SECONDS = 60
MIN_COVERAGE = 54
SENTINEL = -999.25
TIMESTAMP_FORMAT = "%Y/%m/%d %H:%M:%S"
DATE_COLUMN = "YYYY/MM/DD"
TIME_COLUMN = "HH:MM:SS"

RAW_COLUMNS = {
    "ROP": "Rate Of Penetration (ft_per_hr)",
    "WOB": "Weight on Bit (klbs)",
    "RPM": "Rotary RPM (RPM)",
    "SPP": "Standpipe Pressure (psi)",
    "Torque": "Rotary Torque (kft_lb)",
    "HookLoad": "Hook Load (klbs)",
    "Flow": "Flow (flow_percent)",
    "PumpOut": "Total Pump Output (gal_per_min)",
    "MudVol": "Total Mud Volume (barrels)",
    "BlockHeight": "Block Height (feet)",
    "SPM1": "Pump 1 strokes/min (SPM)",
}
CHANNELS = [*RAW_COLUMNS, "Hydraulic_P", "Mech_P"]
FEATURE_COLUMNS = [f"{channel}_{stat}" for channel in CHANNELS for stat in ("mean", "std", "delta")]
ONE_SECOND = pd.Timedelta(seconds=1)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def clean_chunk(chunk: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop SPP sentinel rows, then mask sensor sentinels and derive proxies."""
    spp_raw = chunk[RAW_COLUMNS["SPP"]].to_numpy(dtype=np.float64, copy=False)
    keep = ~(spp_raw <= -900.0)
    dropped = int((~keep).sum())
    retained = chunk.loc[keep]

    out = pd.DataFrame(index=retained.index)
    out["timestamp"] = pd.to_datetime(
        retained[DATE_COLUMN].astype(str) + " " + retained[TIME_COLUMN].astype(str),
        format=TIMESTAMP_FORMAT,
        errors="raise",
    )
    for channel, raw_column in RAW_COLUMNS.items():
        values = retained[raw_column].to_numpy(dtype=np.float64, copy=True)
        values[values == SENTINEL] = np.nan
        out[channel] = values

    spp = out["SPP"].to_numpy(dtype=np.float64, copy=False)
    pump_out = out["PumpOut"].to_numpy(dtype=np.float64, copy=False)
    torque = out["Torque"].to_numpy(dtype=np.float64, copy=False)
    rpm = out["RPM"].to_numpy(dtype=np.float64, copy=False)
    # np.maximum preserves NaN, satisfying the missing-input proxy contract.
    out["Hydraulic_P"] = np.maximum(spp, 0.0) * np.maximum(pump_out, 0.0)
    out["Mech_P"] = np.maximum(torque, 0.0) * np.maximum(rpm, 0.0)
    return out.reset_index(drop=True), dropped


def window_features(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return 39 features and finite coverage for a 60 x 13 window."""
    if values.shape != (WINDOW_SECONDS, len(CHANNELS)):
        raise ValueError(f"expected {(WINDOW_SECONDS, len(CHANNELS))}, got {values.shape}")
    coverage = np.isfinite(values).sum(axis=0).astype(int)
    features: list[float] = []
    for index in range(values.shape[1]):
        finite = values[np.isfinite(values[:, index]), index]
        if finite.size == 0:
            features.extend((np.nan, np.nan, np.nan))
        else:
            features.extend((float(finite.mean()), float(finite.std(ddof=0)), float(finite[-1] - finite[0])))
    return np.asarray(features, dtype=np.float64), coverage


class WindowAccumulator:
    """Keep only one current tumbling window while preserving segment boundaries."""

    def __init__(self) -> None:
        self.previous_time: pd.Timestamp | None = None
        self.segment_id = -1
        self.buffer: list[tuple[pd.Timestamp, np.ndarray]] = []
        self.window_id = 0
        self.metadata: list[dict[str, Any]] = []
        self.feature_rows: list[dict[str, Any]] = []
        self.segment_resets = 0
        self.tail_discarded_rows = 0

    def _discard_tail(self) -> None:
        self.tail_discarded_rows += len(self.buffer)
        self.buffer.clear()

    def _emit_window(self) -> None:
        timestamps = [row[0] for row in self.buffer]
        values = np.vstack([row[1] for row in self.buffer])
        features, coverage = window_features(values)
        bad_channels = [CHANNELS[i] for i, count in enumerate(coverage) if count < MIN_COVERAGE]
        valid = not bad_channels and bool(np.isfinite(features).all())
        reject_reason = "" if valid else (
            "coverage_below_54:" + ",".join(bad_channels) if bad_channels else "non_finite_feature"
        )
        record: dict[str, Any] = {
            "window_id": self.window_id,
            "segment_id": self.segment_id,
            "start_time": timestamps[0].strftime(TIMESTAMP_FORMAT),
            "end_time": timestamps[-1].strftime(TIMESTAMP_FORMAT),
            "valid": valid,
            "reject_reason": reject_reason,
            "n_rows": len(self.buffer),
        }
        record.update({f"{channel}_coverage": int(coverage[i]) for i, channel in enumerate(CHANNELS)})
        self.metadata.append(record)
        if valid:
            self.feature_rows.append({"window_id": self.window_id, **dict(zip(FEATURE_COLUMNS, features.tolist(), strict=True))})
        self.window_id += 1
        self.buffer.clear()

    def ingest(self, cleaned: pd.DataFrame) -> None:
        for timestamp, row in zip(
            cleaned["timestamp"].tolist(),
            cleaned[CHANNELS].to_numpy(dtype=np.float64, copy=False),
            strict=True,
        ):
            if self.previous_time is None:
                self.segment_id += 1
            elif timestamp - self.previous_time != ONE_SECOND:
                self._discard_tail()
                self.segment_id += 1
                self.segment_resets += 1
            self.buffer.append((timestamp, row.copy()))
            self.previous_time = timestamp
            if len(self.buffer) == WINDOW_SECONDS:
                self._emit_window()

    def close(self) -> None:
        self._discard_tail()


def build_outputs(raw_path: Path, output_root: Path) -> dict[str, Any]:
    started = time.monotonic()
    processed_dir = output_root / "data" / "processed"
    manifest_dir = output_root / "outputs" / "manifests"
    processed_dir.mkdir(parents=True, exist_ok=True)
    manifest_dir.mkdir(parents=True, exist_ok=True)

    usecols = [DATE_COLUMN, TIME_COLUMN, *RAW_COLUMNS.values()]
    dtypes = {raw_column: np.float64 for raw_column in RAW_COLUMNS.values()}
    accumulator = WindowAccumulator()
    raw_rows = 0
    dropped_spp = 0
    retained_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(raw_path, usecols=usecols, dtype=dtypes, chunksize=CHUNKSIZE), start=1
    ):
        raw_rows += len(chunk)
        cleaned, dropped = clean_chunk(chunk)
        dropped_spp += dropped
        retained_rows += len(cleaned)
        accumulator.ingest(cleaned)
        print(
            f"chunk={chunk_number} raw_rows={raw_rows} retained_rows={retained_rows} "
            f"dropped_spp={dropped_spp} candidate_windows={accumulator.window_id}",
            flush=True,
        )
    accumulator.close()

    metadata = pd.DataFrame(accumulator.metadata)
    metadata_columns = [
        "window_id", "segment_id", "start_time", "end_time", "valid", "reject_reason", "n_rows",
        *[f"{channel}_coverage" for channel in CHANNELS],
    ]
    metadata = metadata.reindex(columns=metadata_columns)
    features = pd.DataFrame(accumulator.feature_rows).reindex(columns=["window_id", *FEATURE_COLUMNS])
    if not features.empty and not np.isfinite(features[FEATURE_COLUMNS].to_numpy(dtype=np.float64)).all():
        raise RuntimeError("valid feature matrix contains NaN or Inf")

    feature_path = processed_dir / "window_features.csv"
    metadata_path = processed_dir / "window_metadata.csv"
    features.to_csv(feature_path, index=False)
    metadata.to_csv(metadata_path, index=False)

    valid_windows = int(metadata["valid"].sum()) if not metadata.empty else 0
    candidate_windows = len(metadata)
    manifest = {
        "phase": "A",
        "raw_path": str(raw_path),
        "raw_sha256": sha256_file(raw_path),
        "chunksize": CHUNKSIZE,
        "raw_rows": raw_rows,
        "dropped_spp_rows": dropped_spp,
        "retained_rows": retained_rows,
        "segments": accumulator.segment_id + 1 if accumulator.segment_id >= 0 else 0,
        "segment_resets": accumulator.segment_resets,
        "tail_discarded_rows": accumulator.tail_discarded_rows,
        "candidate_windows": candidate_windows,
        "valid_windows": valid_windows,
        "rejected_windows": candidate_windows - valid_windows,
        "feature_columns": FEATURE_COLUMNS,
        "feature_matrix_has_only_finite_values": True,
        "output_sha256": {
            "window_features.csv": sha256_file(feature_path),
            "window_metadata.csv": sha256_file(metadata_path),
        },
        "runtime_seconds": round(time.monotonic() - started, 3),
    }
    manifest_path = manifest_dir / "phase_a_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(
        "PASS "
        f"raw_rows={raw_rows} dropped_spp={dropped_spp} segments={manifest['segments']} "
        f"candidate_windows={candidate_windows} valid_windows={valid_windows} "
        f"rejected_windows={manifest['rejected_windows']} finite_features=true",
        flush=True,
    )
    return manifest


def main() -> None:
    root = Path(__file__).resolve().parents[1]
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--raw",
        type=Path,
        default=root.parent / "Datmin-new" / "dataset" / "56-32 1sec data 27029986.csv",
    )
    parser.add_argument("--output-root", type=Path, default=root)
    args = parser.parse_args()
    build_outputs(args.raw.resolve(), args.output_root.resolve())


if __name__ == "__main__":
    main()


### Phase B Module: `clean-pipeline-56-32/scripts/split_scale.py`
Implements 70:30 chronological split, 1-hour embargo buffer, Train-only fitting of StandardScaler, RobustScaler, and MinMaxScaler with immutability assertion, and Train-only EDA generation.


In [ ]:
%%writefile clean-pipeline-56-32/scripts/split_scale.py
#!/usr/bin/env python3
"""Phase B: chronological split, train-only scaling, and Train-only EDA."""
from __future__ import annotations

import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

ROOT = Path(__file__).resolve().parents[1]

N_VALID = 38_177
N_TRAIN_NOMINAL = 26_723
N_TEST_NOMINAL = N_VALID - N_TRAIN_NOMINAL
EMBARGO_SECONDS = 3_600
TIME_FORMAT = "%Y/%m/%d %H:%M:%S"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _json_value(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def scaler_state(scaler: Any) -> tuple[tuple[str, Any], ...]:
    """Exact, serialization-independent snapshot of all fitted scaler attributes."""
    state: list[tuple[str, Any]] = []
    for name, value in sorted(vars(scaler).items()):
        if isinstance(value, np.ndarray):
            frozen: Any = (value.dtype.str, value.shape, value.tobytes())
        elif isinstance(value, np.generic):
            frozen = (value.dtype.str, value.tobytes())
        else:
            frozen = value
        state.append((name, frozen))
    return tuple(state)


def scaler_parameters(scaler: Any, names: tuple[str, ...]) -> dict[str, Any]:
    return {name: _json_value(getattr(scaler, name)) for name in names}


def strict_valid_mask(values: pd.Series) -> pd.Series:
    parsed = values.map({True: True, False: False, "True": True, "False": False})
    if parsed.isna().any():
        raise ValueError("metadata valid column must contain only True or False")
    return parsed.astype(bool)


def load_valid_join(root: Path) -> tuple[pd.DataFrame, list[str]]:
    manifest_path = root / "outputs/manifests/phase_a_manifest.json"
    features_path = root / "data/processed/window_features.csv"
    metadata_path = root / "data/processed/window_metadata.csv"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_hashes = manifest["output_sha256"]
    for path in (features_path, metadata_path):
        actual = sha256_file(path)
        if actual != expected_hashes[path.name]:
            raise RuntimeError(f"Phase A SHA-256 mismatch for {path.name}: {actual}")

    feature_columns = manifest["feature_columns"]
    if len(feature_columns) != 39 or len(set(feature_columns)) != 39:
        raise RuntimeError("Phase A manifest must declare exactly 39 unique feature columns")
    features = pd.read_csv(features_path)
    metadata = pd.read_csv(metadata_path)
    if features.columns.tolist() != ["window_id", *feature_columns]:
        raise RuntimeError("window_features.csv does not match Phase A feature contract")
    if metadata["window_id"].duplicated().any() or features["window_id"].duplicated().any():
        raise RuntimeError("window_id must be unique in both Phase A inputs")

    valid = metadata.loc[strict_valid_mask(metadata["valid"])].copy()
    joined = valid.merge(features, on="window_id", how="inner", validate="one_to_one")
    if len(valid) != N_VALID or len(features) != N_VALID or len(joined) != N_VALID:
        raise RuntimeError(
            f"expected exactly {N_VALID} valid one-to-one joined windows; "
            f"valid={len(valid)} features={len(features)} joined={len(joined)}"
        )
    if set(valid["window_id"]) != set(features["window_id"]):
        raise RuntimeError("valid metadata and feature window_id sets differ")
    joined["start_time"] = pd.to_datetime(joined["start_time"], format=TIME_FORMAT, errors="raise")
    joined["end_time"] = pd.to_datetime(joined["end_time"], format=TIME_FORMAT, errors="raise")
    if (joined["end_time"] < joined["start_time"]).any():
        raise RuntimeError("window end_time precedes start_time")
    joined = joined.sort_values(["start_time", "window_id"], kind="stable").reset_index(drop=True)
    return joined, feature_columns


def chronological_membership(joined: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    if len(joined) != N_VALID:
        raise ValueError(f"expected {N_VALID} rows before split")
    train = joined.iloc[:N_TRAIN_NOMINAL].copy()
    test_nominal = joined.iloc[N_TRAIN_NOMINAL:].copy()
    if len(test_nominal) != N_TEST_NOMINAL:
        raise AssertionError("nominal 70:30 split has wrong cardinality")
    train_support_end_exclusive = train["end_time"].max() + pd.Timedelta(seconds=1)
    test_start_minimum = train_support_end_exclusive + pd.Timedelta(seconds=EMBARGO_SECONDS)
    embargo = test_nominal.loc[test_nominal["start_time"] < test_start_minimum].copy()
    test = test_nominal.loc[test_nominal["start_time"] >= test_start_minimum].copy()
    if test.empty:
        raise RuntimeError("embargo removed every nominal test window")
    actual_gap = test["start_time"].min() - train_support_end_exclusive
    if actual_gap < pd.Timedelta(seconds=EMBARGO_SECONDS):
        raise AssertionError(f"embargo violation: actual gap is {actual_gap}")
    return train, embargo, test, train_support_end_exclusive


def write_scaled(path: Path, window_ids: pd.Series, values: np.ndarray, feature_columns: list[str]) -> None:
    pd.DataFrame(values, columns=feature_columns).assign(window_id=window_ids.to_numpy()).loc[
        :, ["window_id", *feature_columns]
    ].to_csv(path, index=False)


def save_eda(
    train: pd.DataFrame,
    feature_columns: list[str],
    train_scaled: dict[str, np.ndarray],
    metrics_dir: Path,
    figures_dir: Path,
) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns

    X = train[feature_columns]
    stats = pd.DataFrame(
        {
            "count": X.count(), "mean": X.mean(), "std_ddof0": X.std(ddof=0), "min": X.min(),
            "q1_25": X.quantile(0.25), "median_50": X.median(), "q3_75": X.quantile(0.75),
            "max": X.max(), "skewness": X.skew(),
        }
    )
    stats.index.name = "feature"
    stats.reset_index().to_csv(metrics_dir / "train_descriptive_stats.csv", index=False)
    pearson, spearman = X.corr(method="pearson"), X.corr(method="spearman")
    pearson.index.name = spearman.index.name = "feature"
    pearson.to_csv(metrics_dir / "train_pearson_corr.csv")
    spearman.to_csv(metrics_dir / "train_spearman_corr.csv")

    sns.set_theme(style="white")
    fig, ax = plt.subplots(figsize=(26, 22))
    sns.heatmap(pearson, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"shrink": 0.7})
    ax.set_title("Train-only Pearson correlation (39 features)")
    fig.tight_layout(); fig.savefig(figures_dir / "eda_correlation_heatmap.png", dpi=180); plt.close(fig)

    channel_mean_columns = [column for column in feature_columns if column.endswith("_mean")]
    fig, ax = plt.subplots(figsize=(18, 8))
    sns.boxplot(data=X[channel_mean_columns], showfliers=False, ax=ax)
    ax.set_yscale("symlog", linthresh=1.0); ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Train-only channel mean (symlog scale)"); ax.set_title("Sensor scale disparity across channel means")
    fig.tight_layout(); fig.savefig(figures_dir / "eda_scale_disparity_boxplots.png", dpi=180); plt.close(fig)

    key_features = [name for name in ("ROP_mean", "SPP_mean", "PumpOut_mean", "Hydraulic_P_mean", "Mech_P_mean") if name in feature_columns]
    indexes = [feature_columns.index(name) for name in key_features]
    scalers = {
        "Unscaled": X[key_features].to_numpy(),
        "Standard": train_scaled["standard"][:, indexes],
        "Robust": train_scaled["robust"][:, indexes],
        "MinMax": train_scaled["minmax"][:, indexes],
    }
    fig, axes = plt.subplots(len(key_features), len(scalers), figsize=(18, 3.2 * len(key_features)), squeeze=False)
    for row, feature in enumerate(key_features):
        for column, (label, array) in enumerate(scalers.items()):
            sns.histplot(array[:, row], bins=50, stat="density", element="step", fill=False, ax=axes[row, column])
            axes[row, column].set_title(f"{feature}: {label}"); axes[row, column].set_ylabel("density")
    fig.suptitle("Train-only feature distributions before and after scaling", y=1.002)
    fig.tight_layout(); fig.savefig(figures_dir / "eda_scaler_comparison_distributions.png", dpi=180); plt.close(fig)


def build_outputs(root: Path = ROOT) -> dict[str, Any]:
    started = time.monotonic()
    membership_dir, scaled_dir = root / "data/splits", root / "data/splits/scaled"
    scaler_dir, metrics_dir = root / "models/scalers", root / "outputs/metrics"
    figures_dir, manifest_dir = root / "outputs/figures", root / "outputs/manifests"
    for directory in (membership_dir, scaled_dir, scaler_dir, metrics_dir, figures_dir, manifest_dir):
        directory.mkdir(parents=True, exist_ok=True)

    joined, feature_columns = load_valid_join(root)
    train, embargo, test, train_support_end_exclusive = chronological_membership(joined)
    membership_path = membership_dir / "membership.csv"
    membership = pd.concat([train.assign(split="TRAIN"), embargo.assign(split="EMBARGO"), test.assign(split="TEST")], ignore_index=True)[["window_id", "segment_id", "start_time", "end_time", "split"]].copy()
    membership["start_time"] = membership["start_time"].dt.strftime(TIME_FORMAT)
    membership["end_time"] = membership["end_time"].dt.strftime(TIME_FORMAT)
    if membership["window_id"].duplicated().any() or len(membership) != len(joined):
        raise AssertionError("membership is not mutually exclusive and exhaustive")
    membership.to_csv(membership_path, index=False)

    X_train = train[feature_columns].to_numpy(dtype=np.float64, copy=True)
    X_test = test[feature_columns].to_numpy(dtype=np.float64, copy=True)
    if X_train.shape != (N_TRAIN_NOMINAL, len(feature_columns)):
        raise AssertionError(f"unexpected train matrix shape: {X_train.shape}")
    scaler_specs = {
        "standard": (StandardScaler(with_mean=True, with_std=True), ("mean_", "var_", "scale_")),
        "robust": (RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0), unit_variance=False), ("center_", "scale_")),
        "minmax": (MinMaxScaler(feature_range=(0.0, 1.0), clip=False), ("data_min_", "data_max_", "data_range_", "scale_", "min_")),
    }
    scaler_summary: dict[str, Any] = {}
    train_scaled_by_name: dict[str, np.ndarray] = {}
    output_paths = [membership_path]
    for name, (scaler, parameter_names) in scaler_specs.items():
        scaler.fit(X_train)
        fit_state = scaler_state(scaler)
        train_scaled, test_scaled = scaler.transform(X_train), scaler.transform(X_test)
        if scaler_state(scaler) != fit_state:
            raise AssertionError(f"{name} transform changed fitted scaler state")
        model_path = scaler_dir / f"{name}_scaler.joblib"
        joblib.dump(scaler, model_path)
        train_path, test_path = scaled_dir / f"train_{name}.csv", scaled_dir / f"test_{name}.csv"
        write_scaled(train_path, train["window_id"], train_scaled, feature_columns)
        write_scaled(test_path, test["window_id"], test_scaled, feature_columns)
        output_paths.extend((model_path, train_path, test_path))
        train_scaled_by_name[name] = train_scaled
        scaler_summary[name] = {"class": type(scaler).__name__, "fit_rows": int(len(X_train)), "parameters": scaler_parameters(scaler, parameter_names)}

    save_eda(train, feature_columns, train_scaled_by_name, metrics_dir, figures_dir)
    output_paths.extend([metrics_dir / "train_descriptive_stats.csv", metrics_dir / "train_pearson_corr.csv", metrics_dir / "train_spearman_corr.csv", figures_dir / "eda_correlation_heatmap.png", figures_dir / "eda_scale_disparity_boxplots.png", figures_dir / "eda_scaler_comparison_distributions.png"])
    output_hashes = {str(path.relative_to(root)): sha256_file(path) for path in output_paths}
    test_start = test["start_time"].min()
    manifest = {
        "phase": "B", "input_phase_a_sha256_verified": True, "valid_rows": int(len(joined)),
        "train_rows": int(len(train)), "embargo_rows": int(len(embargo)), "test_rows": int(len(test)),
        "n_train_nominal": N_TRAIN_NOMINAL, "n_test_nominal": N_TEST_NOMINAL, "embargo_seconds": EMBARGO_SECONDS,
        "train_support_end_exclusive": train_support_end_exclusive.strftime(TIME_FORMAT),
        "test_start_minimum": (train_support_end_exclusive + pd.Timedelta(seconds=EMBARGO_SECONDS)).strftime(TIME_FORMAT),
        "first_test_start": test_start.strftime(TIME_FORMAT),
        "actual_embargo_seconds": float((test_start - train_support_end_exclusive).total_seconds()),
        "scalers": scaler_summary, "feature_columns": feature_columns, "output_sha256": output_hashes,
        "runtime_seconds": round(time.monotonic() - started, 3),
    }
    (manifest_dir / "phase_b_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print("PASS " f"valid_rows={len(joined)} train_rows={len(train)} embargo_rows={len(embargo)} test_rows={len(test)} " f"actual_embargo_seconds={manifest['actual_embargo_seconds']:.0f} runtime_seconds={manifest['runtime_seconds']:.3f}", flush=True)
    return manifest


def main() -> None:
    build_outputs()


if __name__ == "__main__":
    main()


### Phase C Module: `clean-pipeline-56-32/scripts/cluster_evaluate.py`
Implements 126-run evaluation grid (3 scalers × 2 algorithms × 7 k values × 3 seeds) trained strictly on Train and evaluated on both Train and Test. Applies locked NO-WINNER rules (COLLAPSE, SHARP_TEST_DROP, UNSTABLE).


In [ ]:
%%writefile clean-pipeline-56-32/scripts/cluster_evaluate.py
#!/usr/bin/env python3
"""Phase C: train-only K-Means/GMM grid evaluation with strict NO-WINNER rules."""
from __future__ import annotations

import hashlib
import json
import os
import shutil
import struct
import subprocess
import time
import zlib
from itertools import combinations
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture

ROOT = Path(__file__).resolve().parents[1]

FEATURE_COUNT = 39
TRAIN_ROWS = 26_723
TEST_ROWS = 11_394
SILHOUETTE_SAMPLE_SIZE = 5_000
SILHOUETTE_SEED = 42
SCALERS = ("standard", "robust", "minmax")
ALGORITHMS = ("kmeans", "gmm")
K_VALUES = tuple(range(2, 9))
SEEDS = (42, 0, 1)
PRIMARY_SEED = 42
COLLAPSE_PROPORTION = 0.01
KMEANS_MAX_ITER = 300
GMM_MAX_ITER = 300


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def index_hash(indices: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(indices, dtype="<i8").tobytes()).hexdigest()


def deterministic_indices(row_count: int) -> np.ndarray:
    if row_count < SILHOUETTE_SAMPLE_SIZE:
        raise ValueError(f"need at least {SILHOUETTE_SAMPLE_SIZE} rows, got {row_count}")
    return np.random.default_rng(SILHOUETTE_SEED).choice(row_count, size=SILHOUETTE_SAMPLE_SIZE, replace=False)


def collapse_detected(counts: np.ndarray, row_count: int) -> bool:
    """A missing or <1% cluster invalidates the run."""
    return bool(np.any(counts == 0) or np.any(counts / row_count < COLLAPSE_PROPORTION))


def sharp_test_drop(silhouette_train: float, silhouette_test: float, dbi_train: float, dbi_test: float) -> bool:
    return bool(
        silhouette_train - silhouette_test > 0.10
        or dbi_test > 1.50 * dbi_train
    )


def verdict_reasons(primary: dict[str, Any], stability: dict[str, Any]) -> list[str]:
    """Apply only the ordered, locked Phase C NO-WINNER rules."""
    reasons: list[str] = []
    if primary["collapse_train"] or primary["collapse_test"]:
        reasons.append("COLLAPSE")
    if primary["non_convergent"]:
        reasons.append("NON_CONVERGENT")
    if primary["silhouette_train"] <= 0 or primary["silhouette_test"] <= 0:
        reasons.append("NEGATIVE_SILHOUETTE")
    if sharp_test_drop(
        primary["silhouette_train"], primary["silhouette_test"], primary["dbi_train"], primary["dbi_test"]
    ):
        reasons.append("SHARP_TEST_DROP")
    if stability["unstable"]:
        reasons.append("UNSTABLE")
    return reasons


def read_scaled(root: Path, scaler: str, split: str, expected_rows: int) -> tuple[np.ndarray, np.ndarray, list[str]]:
    path = root / "data/splits/scaled" / f"{split}_{scaler}.csv"
    frame = pd.read_csv(path)
    if frame.columns[0] != "window_id":
        raise RuntimeError(f"{path.name}: window_id must be the first column")
    features = frame.columns[1:].tolist()
    if len(features) != FEATURE_COUNT or len(set(features)) != FEATURE_COUNT:
        raise RuntimeError(f"{path.name}: expected exactly {FEATURE_COUNT} unique features")
    if len(frame) != expected_rows or frame["window_id"].duplicated().any():
        raise RuntimeError(f"{path.name}: unexpected row count or duplicate window_id")
    values = frame.loc[:, features].to_numpy(dtype=np.float64, copy=True)
    if not np.isfinite(values).all():
        raise RuntimeError(f"{path.name}: scaled values must all be finite")
    return frame["window_id"].to_numpy(dtype=np.int64, copy=True), values, features


def load_inputs(root: Path) -> tuple[dict[str, tuple[np.ndarray, np.ndarray]], list[str], dict[str, str]]:
    phase_b_path = root / "outputs/manifests/phase_b_manifest.json"
    phase_b = json.loads(phase_b_path.read_text(encoding="utf-8"))
    if phase_b.get("train_rows") != TRAIN_ROWS or phase_b.get("test_rows") != TEST_ROWS:
        raise RuntimeError("Phase B manifest row counts do not satisfy Phase C contract")
    expected_features = phase_b.get("feature_columns")
    if not isinstance(expected_features, list) or len(expected_features) != FEATURE_COUNT:
        raise RuntimeError("Phase B manifest must declare exactly 39 feature columns")

    loaded: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    source_hashes = {"outputs/manifests/phase_b_manifest.json": sha256_file(phase_b_path)}
    reference_train_ids: np.ndarray | None = None
    reference_test_ids: np.ndarray | None = None
    for scaler in SCALERS:
        train_path = root / "data/splits/scaled" / f"train_{scaler}.csv"
        test_path = root / "data/splits/scaled" / f"test_{scaler}.csv"
        train_ids, x_train, columns = read_scaled(root, scaler, "train", TRAIN_ROWS)
        test_ids, x_test, test_columns = read_scaled(root, scaler, "test", TEST_ROWS)
        if columns != expected_features or test_columns != expected_features:
            raise RuntimeError(f"{scaler}: scaled feature columns do not match Phase B manifest")
        if set(train_ids) & set(test_ids):
            raise RuntimeError(f"{scaler}: Train and Test window_id overlap")
        if reference_train_ids is None:
            reference_train_ids, reference_test_ids = train_ids, test_ids
        elif not np.array_equal(train_ids, reference_train_ids) or not np.array_equal(test_ids, reference_test_ids):
            raise RuntimeError(f"{scaler}: window_id ordering differs across scaler outputs")
        loaded[scaler] = (x_train, x_test)
        source_hashes[str(train_path.relative_to(root))] = sha256_file(train_path)
        source_hashes[str(test_path.relative_to(root))] = sha256_file(test_path)
    return loaded, expected_features, source_hashes


def build_model(algorithm: str, k: int, seed: int) -> Any:
    if algorithm == "kmeans":
        return KMeans(
            n_clusters=k, init="k-means++", n_init=10, max_iter=KMEANS_MAX_ITER,
            tol=1e-4, algorithm="lloyd", random_state=seed,
        )
    if algorithm == "gmm":
        return GaussianMixture(
            n_components=k, covariance_type="full", n_init=3, max_iter=GMM_MAX_ITER,
            tol=1e-3, reg_covar=1e-6, init_params="kmeans", random_state=seed,
        )
    raise ValueError(f"unknown algorithm: {algorithm}")


def split_metrics(x: np.ndarray, labels: np.ndarray, k: int, sample_indices: np.ndarray) -> tuple[float, float, np.ndarray, np.ndarray, bool]:
    counts = np.bincount(labels, minlength=k).astype(np.int64, copy=False)
    proportions = counts / len(labels)
    # Metrics need at least two observed labels; collapse is still recorded and verdicted.
    observed = np.unique(labels).size
    if observed < 2:
        return float("nan"), float("nan"), counts, proportions, True
    dbi = float(davies_bouldin_score(x, labels))
    sampled_labels = labels[sample_indices]
    silhouette = float("nan") if np.unique(sampled_labels).size < 2 else float(silhouette_score(x[sample_indices], sampled_labels))
    return silhouette, dbi, counts, proportions, collapse_detected(counts, len(labels))


def occupancy_columns(row: dict[str, Any], split: str, counts: np.ndarray, proportions: np.ndarray) -> None:
    for cluster in range(8):
        row[f"{split}_C{cluster}_count"] = int(counts[cluster]) if cluster < len(counts) else 0
        row[f"{split}_C{cluster}_prop"] = float(proportions[cluster]) if cluster < len(proportions) else 0.0


def run_one(
    scaler: str, algorithm: str, k: int, seed: int, x_train: np.ndarray, x_test: np.ndarray,
    train_sample: np.ndarray, test_sample: np.ndarray,
) -> tuple[dict[str, Any], np.ndarray, np.ndarray]:
    model = build_model(algorithm, k, seed)
    started = time.monotonic()
    model.fit(x_train)  # Anti-leakage: only Train is fitted.
    labels_train = model.predict(x_train)
    labels_test = model.predict(x_test)
    silhouette_train, dbi_train, train_counts, train_props, collapse_train = split_metrics(x_train, labels_train, k, train_sample)
    silhouette_test, dbi_test, test_counts, test_props, collapse_test = split_metrics(x_test, labels_test, k, test_sample)
    if algorithm == "kmeans":
        n_iter = int(model.n_iter_)
        converged = n_iter < KMEANS_MAX_ITER
        inertia = float(model.inertia_)
    else:
        n_iter = int(model.n_iter_)
        converged = bool(model.converged_)
        inertia = float("nan")
    row: dict[str, Any] = {
        "scaler": scaler, "algorithm": algorithm, "k": k, "seed": seed,
        "fit_scope": "train_only", "train_rows": len(x_train), "test_rows": len(x_test),
        "silhouette_train": silhouette_train, "silhouette_test": silhouette_test,
        "dbi_train": dbi_train, "dbi_test": dbi_test,
        "collapse_train": collapse_train, "collapse_test": collapse_test,
        "converged": converged, "non_convergent": not converged, "n_iter": n_iter,
        "inertia_train": inertia, "fit_seconds": round(time.monotonic() - started, 6),
    }
    occupancy_columns(row, "train", train_counts, train_props)
    occupancy_columns(row, "test", test_counts, test_props)
    return row, labels_train, labels_test


def stability_summary(records: list[dict[str, Any]]) -> dict[str, Any]:
    if len(records) != len(SEEDS) or {row["seed"] for row in records} != set(SEEDS):
        raise RuntimeError("each ledger configuration requires exactly seeds 42, 0, and 1")
    metrics = {}
    for split in ("train", "test"):
        silhouettes = np.array([row[f"silhouette_{split}"] for row in records], dtype=float)
        dbis = np.array([row[f"dbi_{split}"] for row in records], dtype=float)
        metrics[f"silhouette_range_{split}"] = float(np.ptp(silhouettes))
        median_dbi = float(np.median(dbis))
        metrics[f"dbi_relative_range_{split}"] = float((np.max(dbis) - np.min(dbis)) / median_dbi) if median_dbi > 0 else float("inf")
        aris = [
            float(adjusted_rand_score(left[f"labels_{split}"], right[f"labels_{split}"]))
            for left, right in combinations(records, 2)
        ]
        metrics[f"min_pairwise_ari_{split}"] = min(aris)
    unstable = bool(
        max(metrics["silhouette_range_train"], metrics["silhouette_range_test"]) > 0.05
        or max(metrics["dbi_relative_range_train"], metrics["dbi_relative_range_test"]) > 0.20
        or min(metrics["min_pairwise_ari_train"], metrics["min_pairwise_ari_test"]) < 0.80
    )
    metrics["unstable"] = unstable
    return metrics


def build_ledger(run_rows: list[dict[str, Any]], label_cache: dict[tuple[str, str, int, int], tuple[np.ndarray, np.ndarray]]) -> pd.DataFrame:
    ledger: list[dict[str, Any]] = []
    for scaler in SCALERS:
        for algorithm in ALGORITHMS:
            for k in K_VALUES:
                configuration = [
                    dict(row, labels_train=label_cache[(scaler, algorithm, k, row["seed"])][0], labels_test=label_cache[(scaler, algorithm, k, row["seed"])][1])
                    for row in run_rows if (row["scaler"], row["algorithm"], row["k"]) == (scaler, algorithm, k)
                ]
                stability = stability_summary(configuration)
                primary = next(row for row in configuration if row["seed"] == PRIMARY_SEED)
                reasons = verdict_reasons(primary, stability)
                ledger.append({
                    "scaler": scaler, "algorithm": algorithm, "k": k, "primary_seed": PRIMARY_SEED,
                    "verdict": "ELIGIBLE" if not reasons else "NO-WINNER",
                    "reason_codes": ";".join(reasons) if reasons else "PASS_ALL_LOCKED_RULES",
                    "reason_detail": (
                        f"silhouette_range_train={stability['silhouette_range_train']:.6f}; "
                        f"silhouette_range_test={stability['silhouette_range_test']:.6f}; "
                        f"dbi_relative_range_train={stability['dbi_relative_range_train']:.6f}; "
                        f"dbi_relative_range_test={stability['dbi_relative_range_test']:.6f}; "
                        f"min_pairwise_ari_train={stability['min_pairwise_ari_train']:.6f}; "
                        f"min_pairwise_ari_test={stability['min_pairwise_ari_test']:.6f}"
                    ),
                    "primary_silhouette_train": primary["silhouette_train"], "primary_silhouette_test": primary["silhouette_test"],
                    "primary_dbi_train": primary["dbi_train"], "primary_dbi_test": primary["dbi_test"],
                    "primary_collapse_train": primary["collapse_train"], "primary_collapse_test": primary["collapse_test"],
                    "primary_converged": primary["converged"], **stability,
                })
    return pd.DataFrame(ledger)


def _png(path: Path, image: np.ndarray) -> None:
    """Write RGB PNG using only the Python standard library."""
    height, width, channels = image.shape
    assert channels == 3
    raw = b"".join(bytes((0,)) + image[row].tobytes() for row in range(height))
    chunk = lambda kind, payload: struct.pack(">I", len(payload)) + kind + payload + struct.pack(">I", zlib.crc32(kind + payload) & 0xffffffff)
    path.write_bytes(bytes((137, 80, 78, 71, 13, 10, 26, 10)) + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)) + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def _line(canvas: np.ndarray, x0: int, y0: int, x1: int, y1: int, color: tuple[int, int, int]) -> None:
    points = max(abs(x1 - x0), abs(y1 - y0), 1)
    xs = np.rint(np.linspace(x0, x1, points + 1)).astype(int)
    ys = np.rint(np.linspace(y0, y1, points + 1)).astype(int)
    valid = (xs >= 0) & (xs < canvas.shape[1]) & (ys >= 0) & (ys < canvas.shape[0])
    canvas[ys[valid], xs[valid]] = color


def _marker(canvas: np.ndarray, x: int, y: int, color: tuple[int, int, int]) -> None:
    canvas[max(0, y - 3):y + 4, max(0, x - 3):x + 4] = color


def _multi_panel_png(path: Path, series: list[list[tuple[np.ndarray, tuple[int, int, int]]]], rows: int, cols: int) -> None:
    """Minimal, dependency-free multi-panel line chart; blue=Train, orange=Test."""
    height, width, margin = rows * 270 + 50, cols * 390 + 50, 30
    canvas = np.full((height, width, 3), 255, dtype=np.uint8)
    for panel, lines in enumerate(series):
        row, col = divmod(panel, cols)
        left, top = col * 390 + margin, row * 270 + margin
        right, bottom = left + 330, top + 200
        canvas[top:bottom + 1, left] = (30, 30, 30); canvas[bottom, left:right + 1] = (30, 30, 30)
        values = np.concatenate([line[0] for line in lines])
        finite_values = values[np.isfinite(values)]
        if not len(finite_values):
            continue
        ymin, ymax = float(finite_values.min()), float(finite_values.max())
        pad = max((ymax - ymin) * 0.08, 1e-12); ymin -= pad; ymax += pad
        for grid in range(1, 5):
            y = top + (bottom - top) * grid // 5
            canvas[y, left:right + 1] = (225, 225, 225)
        for values, color in lines:
            xs = np.rint(np.linspace(left + 8, right - 8, len(values))).astype(int)
            finite = np.isfinite(values)
            ys = np.zeros(len(values), dtype=int)
            ys[finite] = np.rint(bottom - 8 - (values[finite] - ymin) / (ymax - ymin) * (bottom - top - 16)).astype(int)
            for start in range(len(values) - 1):
                if finite[start] and finite[start + 1]: _line(canvas, int(xs[start]), int(ys[start]), int(xs[start + 1]), int(ys[start + 1]), color)
            for x, y in zip(xs[finite], ys[finite], strict=True): _marker(canvas, int(x), int(y), color)
    _png(path, canvas)


def annotate_chart(path: Path, title: str, panels: list[str]) -> None:
    """Add compact labels with the already-installed ImageMagick, when available."""
    convert = shutil.which("convert")
    if not convert:
        return  # ponytail: unlabeled fallback; add a plotting dependency only if native convert disappears.
    args = [convert, str(path), "-background", "white", "-gravity", "north", "-splice", "0x32", "-gravity", "northwest", "-fill", "black", "-pointsize", "15", "-annotate", "+10+18", title]
    for index, label in enumerate(panels):
        row, col = divmod(index, 3)
        args.extend(["-pointsize", "12", "-annotate", f"+{35 + col * 390}+{45 + row * 270}", label])
    args.extend(["-gravity", "south", "-splice", "0x22", "-pointsize", "12", "-annotate", "+0+5", "x: k = 2, 3, 4, 5, 6, 7, 8   |   blue: Train   orange: Test", str(path)])
    subprocess.run(args, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def save_figures(primary: pd.DataFrame, figures_dir: Path) -> None:
    # stdlib fallback: panel order is KMEANS/GMM (rows) × standard/robust/minmax (columns).
    for metric, filename in (("silhouette", "cluster_silhouette_train_vs_test.png"), ("dbi", "cluster_dbi_train_vs_test.png")):
        panels = []
        for algorithm in ALGORITHMS:
            for scaler in SCALERS:
                subset = primary.loc[(primary["algorithm"] == algorithm) & (primary["scaler"] == scaler)].sort_values("k")
                panels.append([(subset[f"{metric}_train"].to_numpy(), (31, 119, 180)), (subset[f"{metric}_test"].to_numpy(), (255, 127, 14))])
        path = figures_dir / filename
        _multi_panel_png(path, panels, rows=2, cols=3)
        annotate_chart(path, f"{metric.upper()} — Train vs Test", ["KMeans · standard", "KMeans · robust", "KMeans · minmax", "GMM · standard", "GMM · robust", "GMM · minmax"])
    inertia_panels = []
    for scaler in SCALERS:
        subset = primary.loc[(primary["algorithm"] == "kmeans") & (primary["scaler"] == scaler)].sort_values("k")
        inertia_panels.append([(subset["inertia_train"].to_numpy(), (31, 119, 180))])
    path = figures_dir / "cluster_kmeans_inertia_elbow.png"
    _multi_panel_png(path, inertia_panels, rows=1, cols=3)
    annotate_chart(path, "K-MEANS TRAIN INERTIA — Elbow", ["standard", "robust", "minmax"])


def build_outputs(root: Path = ROOT) -> dict[str, Any]:
    started = time.monotonic()
    metrics_dir, figures_dir, manifest_dir = root / "outputs/metrics", root / "outputs/figures", root / "outputs/manifests"
    for directory in (metrics_dir, figures_dir, manifest_dir):
        directory.mkdir(parents=True, exist_ok=True)
    inputs, feature_columns, source_hashes = load_inputs(root)
    train_sample, test_sample = deterministic_indices(TRAIN_ROWS), deterministic_indices(TEST_ROWS)

    run_rows: list[dict[str, Any]] = []
    label_cache: dict[tuple[str, str, int, int], tuple[np.ndarray, np.ndarray]] = {}
    for scaler in SCALERS:
        x_train, x_test = inputs[scaler]
        for algorithm in ALGORITHMS:
            for k in K_VALUES:
                for seed in SEEDS:
                    row, labels_train, labels_test = run_one(scaler, algorithm, k, seed, x_train, x_test, train_sample, test_sample)
                    run_rows.append(row)
                    label_cache[(scaler, algorithm, k, seed)] = (labels_train, labels_test)
                    print(
                        f"RUN scaler={scaler} algorithm={algorithm} k={k} seed={seed} "
                        f"sil_train={row['silhouette_train']:.6f} sil_test={row['silhouette_test']:.6f} "
                        f"dbi_train={row['dbi_train']:.6f} dbi_test={row['dbi_test']:.6f}", flush=True,
                    )
    if len(run_rows) != 126:
        raise AssertionError(f"expected 126 runs, got {len(run_rows)}")
    all_results = pd.DataFrame(run_rows)
    primary = all_results.loc[all_results["seed"] == PRIMARY_SEED].copy()
    ledger = build_ledger(run_rows, label_cache)
    if len(primary) != 42 or len(ledger) != 42:
        raise AssertionError("primary and verdict row counts must each be 42")

    all_path = metrics_dir / "clustering_results_all.csv"
    primary_path = metrics_dir / "clustering_summary_primary42.csv"
    ledger_path = metrics_dir / "verdict_ledger.csv"
    all_results.to_csv(all_path, index=False)
    primary.to_csv(primary_path, index=False)
    ledger.to_csv(ledger_path, index=False)
    save_figures(primary, figures_dir)
    figure_paths = [
        figures_dir / "cluster_silhouette_train_vs_test.png", figures_dir / "cluster_dbi_train_vs_test.png",
        figures_dir / "cluster_kmeans_inertia_elbow.png",
    ]
    output_paths = [all_path, primary_path, ledger_path, *figure_paths]
    verdict_counts = ledger["verdict"].value_counts().to_dict()
    manifest = {
        "phase": "C", "run_count": len(run_rows), "primary_seed_run_count": len(primary), "verdict_count": len(ledger),
        "feature_count": len(feature_columns), "fit_scope": "train_only", "train_rows": TRAIN_ROWS, "test_rows": TEST_ROWS,
        "grid": {"scalers": list(SCALERS), "algorithms": list(ALGORITHMS), "k_values": list(K_VALUES), "seeds": list(SEEDS)},
        "silhouette": {"sample_size": SILHOUETTE_SAMPLE_SIZE, "seed": SILHOUETTE_SEED, "train_indices_sha256": index_hash(train_sample), "test_indices_sha256": index_hash(test_sample)},
        "thresholds": {"collapse_proportion_lt": COLLAPSE_PROPORTION, "silhouette_range_gt": 0.05, "dbi_relative_range_gt": 0.20, "pairwise_ari_lt": 0.80},
        "status_counts": {key: int(value) for key, value in verdict_counts.items()},
        "primary_collapse_runs": int((primary["collapse_train"] | primary["collapse_test"]).sum()),
        "primary_non_convergent_runs": int(primary["non_convergent"].sum()),
        "input_sha256": source_hashes, "output_sha256": {str(path.relative_to(root)): sha256_file(path) for path in output_paths},
        "runtime_seconds": round(time.monotonic() - started, 3),
    }
    manifest_path = manifest_dir / "phase_c_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(
        f"PASS phase=C runs={len(run_rows)} verdicts={len(ledger)} "
        f"eligible={manifest['status_counts'].get('ELIGIBLE', 0)} "
        f"no_winner={manifest['status_counts'].get('NO-WINNER', 0)} runtime_seconds={manifest['runtime_seconds']:.3f}",
        flush=True,
    )
    return manifest


def main() -> None:
    build_outputs()


if __name__ == "__main__":
    main()


### Contract Checks Module: `clean-pipeline-56-32/scripts/check_contract.py`
Assert-based synthetic tests covering sentinels, 54/60 threshold, feature math, chunk invariance, gap reset, tumbling anchors, embargo spacing, scaler state immutability, and verdict rule hierarchy.


In [ ]:
%%writefile clean-pipeline-56-32/scripts/check_contract.py
#!/usr/bin/env python3
"""Assert-based synthetic contract checks for Phase A windowing."""
from __future__ import annotations

import numpy as np
import pandas as pd

from prepare_windows import (
    CHANNELS,
    DATE_COLUMN,
    FEATURE_COLUMNS,
    RAW_COLUMNS,
    SENTINEL,
    TIME_COLUMN,
    WINDOW_SECONDS,
    WindowAccumulator,
    clean_chunk,
    window_features,
)
from cluster_evaluate import collapse_detected, sharp_test_drop, verdict_reasons
from split_scale import EMBARGO_SECONDS, scaler_state
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler


def cleaned_rows(start: str, count: int, value: float = 1.0) -> pd.DataFrame:
    timestamps = pd.date_range(start, periods=count, freq="s")
    data = {"timestamp": timestamps}
    for index, channel in enumerate(CHANNELS):
        data[channel] = np.arange(count, dtype=np.float64) + value + index * 100.0
    return pd.DataFrame(data)


def raw_rows() -> pd.DataFrame:
    times = pd.date_range("2021-02-08 04:30:00", periods=3, freq="s")
    data: dict[str, object] = {
        DATE_COLUMN: times.strftime("%Y/%m/%d"),
        TIME_COLUMN: times.strftime("%H:%M:%S"),
    }
    for raw_column in RAW_COLUMNS.values():
        data[raw_column] = np.array([10.0, 20.0, 30.0])
    data[RAW_COLUMNS["SPP"]] = np.array([100.0, -900.0, 100.0])
    data[RAW_COLUMNS["ROP"]] = np.array([SENTINEL, 2.0, 3.0])
    return pd.DataFrame(data)


def emitted(rows: pd.DataFrame, splits: list[int]) -> WindowAccumulator:
    accumulator = WindowAccumulator()
    cursor = 0
    for stop in splits:
        accumulator.ingest(rows.iloc[cursor:stop].reset_index(drop=True))
        cursor = stop
    accumulator.close()
    return accumulator


def main() -> None:
    # SPP <= -900 is dropped before masking; exact sentinel is NaN in other sensors.
    cleaned, dropped = clean_chunk(raw_rows())
    assert dropped == 1 and len(cleaned) == 2
    assert np.isnan(cleaned.loc[0, "ROP"]), "sentinel -999.25 must become NaN"

    # A 54/60 coverage channel is accepted; 53/60 is rejected.
    values = cleaned_rows("2021-01-01", WINDOW_SECONDS)[CHANNELS].to_numpy(dtype=np.float64, copy=True)
    values[:6, 0] = np.nan
    features, coverage = window_features(values)
    assert coverage[0] == 54 and len(features) == 39
    assert np.isfinite(features).all()
    values[6, 0] = np.nan
    _, coverage = window_features(values)
    assert coverage[0] == 53

    # Exact feature contract: mean, population std (ddof=0), endpoint delta, 13 x 3.
    simple = np.tile(np.arange(WINDOW_SECONDS, dtype=np.float64)[:, None], (1, len(CHANNELS)))
    features, coverage = window_features(simple)
    assert FEATURE_COLUMNS[:3] == ["ROP_mean", "ROP_std", "ROP_delta"]
    assert len(FEATURE_COLUMNS) == 39 and len(features) == 39
    assert np.all(coverage == WINDOW_SECONDS)
    assert np.isclose(features[0], 29.5)
    assert np.isclose(features[1], np.std(np.arange(WINDOW_SECONDS), ddof=0))
    assert np.isclose(features[2], 59.0)

    # Chunk boundaries cannot change emitted windows or feature values.
    rows = cleaned_rows("2021-01-01", 120)
    one_chunk = emitted(rows, [120])
    split_chunks = emitted(rows, [37, 84, 120])
    assert one_chunk.metadata == split_chunks.metadata
    assert one_chunk.feature_rows == split_chunks.feature_rows

    # A non-1-second timestamp gap resets the segment and discards both partial tails.
    before_gap = cleaned_rows("2021-01-01", 30)
    after_gap = cleaned_rows("2021-01-01 00:00:31", 30)
    gap_case = emitted(pd.concat([before_gap, after_gap], ignore_index=True), [60])
    assert gap_case.window_id == 0 and gap_case.segment_id == 1
    assert gap_case.segment_resets == 1 and gap_case.tail_discarded_rows == 60

    # Tumbling anchors are the first retained row of each 60-row interval.
    tumbling = emitted(rows, [120])
    assert [item["start_time"] for item in tumbling.metadata] == [
        "2021/01/01 00:00:00", "2021/01/01 00:01:00"
    ]
    assert [item["end_time"] for item in tumbling.metadata] == [
        "2021/01/01 00:00:59", "2021/01/01 00:01:59"
    ]

    # Phase B: Test must start at least one hour after inclusive Train support.
    train_support_end_exclusive = pd.Timestamp("2021-01-01 10:00:59") + pd.Timedelta(seconds=1)
    first_test_start = train_support_end_exclusive + pd.Timedelta(seconds=EMBARGO_SECONDS)
    assert (first_test_start - train_support_end_exclusive).total_seconds() >= EMBARGO_SECONDS

    # Phase B: membership assigns every window exactly once.
    membership = pd.DataFrame({"window_id": [10, 11, 12, 13], "split": ["TRAIN", "TRAIN", "EMBARGO", "TEST"]})
    expected_ids = {10, 11, 12, 13}
    assert not membership["window_id"].duplicated().any()
    assert set(membership["window_id"]) == expected_ids
    assert set(membership["split"]) == {"TRAIN", "EMBARGO", "TEST"}

    # Phase B: transform of arbitrary Test-like data cannot mutate Train-fitted state.
    train_matrix = np.arange(24, dtype=np.float64).reshape(8, 3)
    dummy_test = np.array([[1e9, -1e9, 42.0], [-3e9, 9e9, -7.0]], dtype=np.float64)
    for scaler in (
        StandardScaler(with_mean=True, with_std=True),
        RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0), unit_variance=False),
        MinMaxScaler(feature_range=(0.0, 1.0), clip=False),
    ):
        scaler.fit(train_matrix)
        before = scaler_state(scaler)
        scaler.transform(dummy_test)
        assert scaler_state(scaler) == before, f"{type(scaler).__name__} state changed during Test transform"

    # Phase C: any empty or <1% cluster is COLLAPSE (e.g., 267 of 26,723).
    assert collapse_detected(np.array([267, 26_456]), 26_723)
    assert not collapse_detected(np.array([268, 26_455]), 26_723)

    # Phase C: strict sharp generalization drop applies to either locked condition.
    assert sharp_test_drop(0.45, 0.34, 0.80, 0.90)
    assert sharp_test_drop(0.45, 0.40, 0.80, 1.21)
    assert not sharp_test_drop(0.45, 0.36, 0.80, 1.20)

    # Phase C: NO-WINNER reasons cannot be bypassed; only a fully clean primary is eligible.
    stable = {"unstable": False}
    clean = {"collapse_train": False, "collapse_test": False, "non_convergent": False, "silhouette_train": 0.20, "silhouette_test": 0.15, "dbi_train": 1.0, "dbi_test": 1.2}
    assert verdict_reasons(clean, stable) == []
    assert verdict_reasons({**clean, "collapse_test": True}, stable) == ["COLLAPSE"]
    assert verdict_reasons({**clean, "silhouette_test": 0.0}, stable) == ["NEGATIVE_SILHOUETTE", "SHARP_TEST_DROP"]
    assert verdict_reasons(clean, {"unstable": True}) == ["UNSTABLE"]
    print("PASS: Phase A, Phase B, and Phase C contract checks")


if __name__ == "__main__":
    main()


#### Cell: Setup — locate the pipeline, resolve the raw CSV
**What:** finds `clean-pipeline-56-32/scripts` from any working dir, imports the three phase modules, resolves the 357 MB raw CSV (repo → Kaggle Input → kagglehub), and prints the locked protocol.
**Why:** the scripts default `--raw` to the author's laptop path — never rely on it; explicit paths keep runs reproducible anywhere.
**Read this:** printed `CP_ROOT`, `RAW_CSV` (must exist), and the protocol summary.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

def find_cp_root():
    p = Path.cwd()
    for _ in range(6):
        for rel in ['clean-pipeline-56-32', 'datmin/clean-pipeline-56-32']:
            cand = p / rel
            if (cand / 'scripts' / 'check_contract.py').is_file():
                return cand.resolve()
        p = p.parent
    cand = (Path.cwd() / 'clean-pipeline-56-32').resolve()
    (cand / 'scripts').mkdir(parents=True, exist_ok=True)
    return cand

CP_ROOT = find_cp_root()
sys.path.insert(0, str(CP_ROOT / 'scripts'))
import check_contract, prepare_windows, split_scale, cluster_evaluate
print('CP_ROOT:', CP_ROOT)
print('imported from:', Path(check_contract.__file__).parent)

def resolve_raw(csv='56-32 1sec data 27029986.csv'):
    repo = CP_ROOT.parent
    for c in [repo / 'dataset' / csv, Path.cwd() / 'dataset' / csv, Path('/kaggle/input')]:
        if c.is_file():
            return c
        if c.is_dir():
            hits = list(c.rglob(csv))
            if hits:
                return hits[0]
    for d in [Path.cwd(), Path('/content')]:
        if d.is_dir():
            hits = list(d.rglob(csv))
            if hits:
                return hits[0]
    try:
        import kagglehub
        return Path(kagglehub.dataset_download('faruqmahdison/utah-datmin')) / csv
    except Exception as e:
        raise FileNotFoundError('raw CSV not found (repo/Kaggle/kagglehub): ' + str(e))

RAW_CSV = resolve_raw()
print('RAW_CSV:', RAW_CSV)
proto = json.loads((CP_ROOT / 'protocol.json').read_text(encoding='utf-8'))
print('protocol:', proto['protocol_version'], '| windows:', proto['phase_a']['window_seconds'], 's/', proto['phase_a']['stride_seconds'], 's | split:', proto['phase_b']['split'], '| scalers:', proto['phase_b']['candidate_scalers'], '| models:', proto['phase_c']['models'])


#### Cell: Contract checks (no dataset needed)
**What:** runs `check_contract.main()` — synthetic unit tests for sentinel masking, 54/60 coverage edge, feature math, chunk-invariance, gap reset, tumbling anchors, embargo, membership, scaler-state immutability, and verdict rules.
**Why:** prove the machinery correct BEFORE spending 30 minutes on data. A FAIL here stops everything.
**Read this:** single `PASS` line; any AssertionError names the broken contract.


In [ ]:
check_contract.main()


## Phase A — Tumbling windows (`prepare_windows`)
13 channels (11 raw + Hydraulic_P/Mech_P proxies), SPP row-drop, sentinel masking, gap reset on any non-+1s delta, coverage gate ≥54/60, tumbling emit. Expect: 28 segments, 41.271 candidates → 38.177 valid.


In [ ]:
man_a = prepare_windows.build_outputs(RAW_CSV, CP_ROOT)
print('segments:', man_a['segments'], '| resets:', man_a['segment_resets'])
print('candidate:', man_a['candidate_windows'], '| valid:', man_a['valid_windows'], '| rejected:', man_a['rejected_windows'])
print('runtime_s:', man_a['runtime_seconds'])


In [ ]:
import numpy as np
meta = pd.read_csv(CP_ROOT / 'data/processed/window_metadata.csv')
print('reject reasons:')
print(meta[~meta['valid']]['reject_reason'].value_counts().to_string())
feat = pd.read_csv(CP_ROOT / 'data/processed/window_features.csv')
print('features:', feat.shape, '| finite all:', bool(np.isfinite(feat.drop(columns=['window_id']).to_numpy(dtype='float64')).all()))
display(feat.head(3))
display(meta.head(3))


## Phase B — Chronological split + train-only scaling (`split_scale`)
70% awal Train / embargo 1 jam / 30% akhir Test (counts + gap asserted); Standard/Robust/MinMax fit Train-only with state-freeze check; scaler joblibs saved; EDA strictly train-only.


In [ ]:
man_b = split_scale.build_outputs(CP_ROOT)
print('train:', man_b['train_rows'], '| embargo:', man_b['embargo_rows'], '| test:', man_b['test_rows'])
print('actual embargo_s:', round(man_b['actual_embargo_seconds']))
print('scalers fit_rows:', {k: v['fit_rows'] for k, v in man_b['scalers'].items()})


In [ ]:
m = CP_ROOT / 'outputs/metrics'
display(pd.read_csv(m / 'train_descriptive_stats.csv').head(10))
for fig in ['eda_correlation_heatmap.png', 'eda_scale_disparity_boxplots.png', 'eda_scaler_comparison_distributions.png']:
    display(Image(str(CP_ROOT / 'outputs/figures' / fig)))
print('Note vs our EDA: everything above is TRAIN-ONLY (no future leakage); ours is global/strided.')


## Phase C — 126-run grid (`cluster_evaluate`)
3 scalers × KMeans/GMM × k=2..8 × seeds 42/0/1, fit train-only, evaluated on both splits. Rules: COLLAPSE (<1% cluster), SHARP_TEST_DROP (Sil −0.10 / DBI ×1.5), UNSTABLE (Sil-range/DBI-range/ARI), else ELIGIBLE. Expect belasan menit CPU (GMM heavy).


In [ ]:
man_c = cluster_evaluate.build_outputs(CP_ROOT)
print('runs:', man_c['run_count'], '| verdicts:', man_c['verdict_count'])
print('status:', man_c['status_counts'])
print('runtime_s:', man_c['runtime_seconds'])


In [ ]:
v2 = CP_ROOT / 'outputs/metrics'
ledger = pd.read_csv(v2 / 'verdict_ledger.csv')
print('== verdict counts ==')
print(ledger['verdict'].value_counts().to_string())
print('== reason codes ==')
print(ledger['reason_codes'].value_counts().to_string())
print('== robust/kmeans/k4 (our locked choice under their rules) ==')
display(ledger[(ledger['scaler']=='robust')&(ledger['algorithm']=='kmeans')&(ledger['k']==4)][['verdict','reason_codes','primary_silhouette_train','primary_silhouette_test','primary_dbi_train','primary_dbi_test']])
print('== ELIGIBLE rows ==')
display(ledger[ledger['verdict']=='ELIGIBLE'][['scaler','algorithm','k','primary_silhouette_train','primary_silhouette_test','primary_dbi_train','primary_dbi_test']])


In [ ]:
for fig in ['cluster_silhouette_train_vs_test.png', 'cluster_dbi_train_vs_test.png', 'cluster_kmeans_inertia_elbow.png']:
    display(Image(str(CP_ROOT / 'outputs/figures' / fig)))
print('blue=Train, orange=Test. Watch train→test drops (non-stationarity) and flat-vs-elbow shapes.')


## Recap — what each artifact proves + open team questions
- Contract PASS → machinery trustworthy. Phase A manifest → window attrition accounted. Phase B manifest → split/embargo/scaler discipline. Ledger → only 4/42 ELIGIBLE; robust/kmeans/k4 NO-WINNER(COLLAPSE) vs our gate-PASS winner: same data, different verdict philosophy (collapse-averse vs idle-tolerant).
- Open: (1) 7 sensors vs 13 channels, (2) verdict philosophy for the paper, (3) regime claims need driller-log validation per protocol.
- Run next cell: MISSING-check + export zip.


In [ ]:
need = ['outputs/manifests/phase_a_manifest.json','outputs/manifests/phase_b_manifest.json','outputs/manifests/phase_c_manifest.json','outputs/metrics/verdict_ledger.csv','outputs/metrics/clustering_summary_primary42.csv','outputs/metrics/train_descriptive_stats.csv','outputs/figures/cluster_silhouette_train_vs_test.png','outputs/figures/cluster_dbi_train_vs_test.png','outputs/figures/cluster_kmeans_inertia_elbow.png','outputs/figures/eda_correlation_heatmap.png']
missing=[f for f in need if not (CP_ROOT/f).exists()]
print('MISSING:', missing if missing else 'none — all clean-pipeline artifacts present')
try:
    import subprocess as _sp
    _rev = _sp.run(['git', '-C', str(CP_ROOT.parent), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
    print('repo HEAD:', _rev if _rev else 'n/a (not a git checkout)')
except Exception as e:
    print('repo HEAD: n/a -', e)


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT = CP_ROOT / 'outputs'
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
print('OUT dir (absolute):', OUT.resolve())
zp = live / 'clean_pipeline_walkthrough_outputs.zip'
count = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in ('.csv', '.json', '.png'):
            z.write(p, p.relative_to(OUT)); count += 1
print('packed:', zp.resolve(), '| files:', count)
if Path('/kaggle/output').exists():
    import shutil as _sh
    pub = Path('/kaggle/output') / 'clean_pipeline_walkthrough'
    pub.mkdir(parents=True, exist_ok=True)
    _sh.copy2(zp, pub / zp.name)
    for extra in ['metrics/verdict_ledger.csv','metrics/clustering_summary_primary42.csv','manifests/phase_a_manifest.json','manifests/phase_b_manifest.json','manifests/phase_c_manifest.json']:
        s = OUT / extra
        if s.exists():
            _sh.copy2(s, pub / Path(extra).name)
    print('mirrored key artifacts to (persisted):', pub.resolve())
try:
    import google.colab
    from google.colab import files
    files.download(str(zp)); print('browser download started.')
except Exception:
    print('no Colab download; zip at:', zp.resolve())
